# Training Data Splitting

Create dataset splits for:
- **Past:** 1850–1950
- **Present:** 1950–2015
- **Future:** 2015–2100
- **Hemispheres:** Northern, Equator band, Southern

In [1]:
import numpy as np
import xarray as xr

x_data = xr.open_dataset("../../data/CanESM_1850-2100_tas.nc", engine="netcdf4")
y_data_rsut = xr.open_dataset("../../data/CanESM_1850-2100_rsutcre.nc", engine="netcdf4")
y_data_rlut = xr.open_dataset("../../data/CanESM_1850-2100_rlutcre.nc", engine="netcdf4")

def select_var(ds, preferred_names):
    for name in preferred_names:
        if name in ds.data_vars:
            return name
    non_bnds = [name for name in ds.data_vars if not name.endswith("_bnds")]
    if not non_bnds:
        raise ValueError("No usable data variable found in dataset")
    return non_bnds[0]

x_var = select_var(x_data, ["tas"])
y_var_rsut = select_var(y_data_rsut, ["cre"])
y_var_rlut = select_var(y_data_rlut, ["cre"] )

x_da = x_data[x_var]
y_da_rsut = y_data_rsut[y_var_rsut]
y_da_rlut = y_data_rlut[y_var_rlut]

x_da, y_da_rsut, y_da_rlut = xr.align(x_da, y_da_rsut, y_da_rlut, join="inner")

print("Loaded and aligned:")
print("x variable:", x_var, "| dims:", x_da.dims)
print("y_rsut variable:", y_var_rsut, "| dims:", y_da_rsut.dims)
print("y_rlut variable:", y_var_rlut, "| dims:", y_da_rlut.dims)

Loaded and aligned:
x variable: tas | dims: ('member', 'time', 'lat', 'lon')
y_rsut variable: cre | dims: ('time', 'lat', 'lon')
y_rlut variable: cre | dims: ('time', 'lat', 'lon')


In [2]:
time_dim_candidates = ["time", "year", "date"]
time_dim = next((d for d in time_dim_candidates if d in x_da.dims and d in y_da_rsut.dims and d in y_da_rlut.dims), None)

if time_dim is None:
    raise ValueError("Could not find a shared time dimension (expected one of: time/year/date).")

years = x_da[time_dim].dt.year

def subset_time(da, dim, mask):
    idx = np.where(mask.values)[0]
    return da.isel({dim: idx})

mask_past = (years >= 1850) & (years < 1950)
mask_present = (years >= 1950) & (years < 2015)
mask_future = (years >= 2015) & (years <= 2100)

X_past = subset_time(x_da, time_dim, mask_past)
X_present = subset_time(x_da, time_dim, mask_present)
X_future = subset_time(x_da, time_dim, mask_future)

y_rsut_past = subset_time(y_da_rsut, time_dim, mask_past)
y_rsut_present = subset_time(y_da_rsut, time_dim, mask_present)
y_rsut_future = subset_time(y_da_rsut, time_dim, mask_future)

y_rlut_past = subset_time(y_da_rlut, time_dim, mask_past)
y_rlut_present = subset_time(y_da_rlut, time_dim, mask_present)
y_rlut_future = subset_time(y_da_rlut, time_dim, mask_future)

time_splits = {
    "past_1850_1950": {"X": X_past, "y_rsut": y_rsut_past, "y_rlut": y_rlut_past},
    "present_1950_2015": {"X": X_present, "y_rsut": y_rsut_present, "y_rlut": y_rlut_present},
    "future_2015_2100": {"X": X_future, "y_rsut": y_rsut_future, "y_rlut": y_rlut_future},
}

print("Time split dimension:", time_dim)
for split_name, split_data in time_splits.items():
    n = split_data["X"].sizes[time_dim]
    print(f"{split_name}: {n} samples")

Time split dimension: time
past_1850_1950: 1200 samples
present_1950_2015: 780 samples
future_2015_2100: 1032 samples


In [3]:
lat_dim_candidates = ["lat", "latitude", "y"]
lat_dim = next((d for d in lat_dim_candidates if d in x_da.dims and d in y_da_rsut.dims and d in y_da_rlut.dims), None)

if lat_dim is None:
    raise ValueError("Could not find a shared latitude dimension (expected one of: lat/latitude/y).")

lat_vals = x_da[lat_dim]

mask_north = lat_vals > 23.5
mask_equator = (lat_vals >= -23.5) & (lat_vals <= 23.5)
mask_south = lat_vals < -23.5

def subset_lat(da, mask):
    return da.where(mask, drop=True)

X_north = subset_lat(x_da, mask_north)
X_equator = subset_lat(x_da, mask_equator)
X_south = subset_lat(x_da, mask_south)

y_rsut_north = subset_lat(y_da_rsut, mask_north)
y_rsut_equator = subset_lat(y_da_rsut, mask_equator)
y_rsut_south = subset_lat(y_da_rsut, mask_south)

y_rlut_north = subset_lat(y_da_rlut, mask_north)
y_rlut_equator = subset_lat(y_da_rlut, mask_equator)
y_rlut_south = subset_lat(y_da_rlut, mask_south)

hemisphere_splits = {
    "northern": {"X": X_north, "y_rsut": y_rsut_north, "y_rlut": y_rlut_north},
    "equator_band": {"X": X_equator, "y_rsut": y_rsut_equator, "y_rlut": y_rlut_equator},
    "southern": {"X": X_south, "y_rsut": y_rsut_south, "y_rlut": y_rlut_south},
}

print("Latitude split dimension:", lat_dim)
for split_name, split_data in hemisphere_splits.items():
    n = split_data["X"].sizes[lat_dim]
    print(f"{split_name}: {n} latitude points")

Latitude split dimension: lat
northern: 24 latitude points
equator_band: 16 latitude points
southern: 24 latitude points
